In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [ ]:
df = pd.read_csv("C:/Sarang/College/lp3/ml/sales_data_sample.csv", encoding="ISO-8859-1")

print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
features = [
    "QUANTITYORDERED",
    "PRICEEACH",
    "SALES",
    "MSRP"
]

X = df[features].copy()

print("\nSelected features:")
print(X.head())

In [ ]:
print("\nMissing values:")
print(X.isnull().sum())

X = X.dropna()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
inertia = []

K = range(1, 11)

for k in K:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(kmeans.inertia_)

In [ ]:
kl = KneeLocator(
    K,
    inertia,
    curve="convex",
    direction="decreasing"
)

optimal_k = kl.elbow

print("\nOptimal number of clusters:", optimal_k)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    K,
    inertia,
    marker="o"
)

if optimal_k is not None:
    plt.axvline(
        x=optimal_k,
        color="red",
        linestyle="--",
        label=f"Optimal K = {optimal_k}"
    )

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method with Automatic Detection")

plt.xticks(K)
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
if optimal_k is None:
    raise ValueError("Elbow point could not be detected.")

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

In [ ]:
X["Cluster"] = clusters

print("\nClustered data:")
print(X.head(10))

In [ ]:
print("\nNumber of records in each cluster:")
print(X["Cluster"].value_counts().sort_index())

In [ ]:
centers = scaler.inverse_transform(kmeans.cluster_centers_)

cluster_centers = pd.DataFrame(
    centers,
    columns=features
)

print("\nCluster Centers:")
print(cluster_centers)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X["QUANTITYORDERED"],
    X["SALES"],
    c=X["Cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Quantity Ordered")
plt.ylabel("Sales")
plt.title("K-Means Clustering of Sales Data")
plt.colorbar(label="Cluster")

plt.show()